In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 설정된 폰트 목록 출력
sys_font = [f.name for f in fm.fontManager.ttflist]
print(sys_font)

# 나눔고딕 폰트가 목록에 있는지 확인
print('NanumGothic' in sys_font)

In [ ]:
!pip install koreanize-matplotlib

In [9]:
import koreanize_matplotlib

# 한국어 형태소 분석 및 데이터 시각화

이 노트북은 Bareun API를 사용하여 한국어 텍스트 데이터에서 명사를 추출하고, 다양한 시각화 기법을 통해 분석하는 방법을 설명합니다.

## 주요 기능
1. Bareun API를 사용한 한국어 형태소 분석
2. 뉴스 데이터에서 명사 추출
3. 워드클라우드, 네트워크 분석, 막대 차트를 통한 시각화

In [ ]:
# Bareun API 패키지 다운로드
# curl 명령을 사용해 Bareun Linux 패키지를 다운로드
# -L: 리다이렉션을 따름, -J: Content-Disposition 헤더 존재 시 파일명 사용, -k: SSL 인증 검증 무시, -s: 진행 상황 표시 없음
!curl -LJks -H "uname:$(uname -a)" https://bareun.ai/api/get -o bareun-linux.deb

# 현재 디렉토리의 파일 목록 표시
!ls

In [ ]:
# 시스템 정보 출력
# uname -a: 운영체제 이름, 호스트명, 커널 버전 등의 시스템 정보 출력
!uname -a

In [ ]:
# 다운로드한 Bareun 패키지 설치
# dpkg -i: 데비안 패키지 설치
!dpkg -i bareun-linux.deb

In [ ]:
# Bareun 환경 변수 설정
# %env: Jupyter 매직 명령어로 환경 변수 설정
%env BAREUN_ROOT="/opt/bareun"  # Bareun 설치 루트 경로 설정
%env LD_LIBRARY_PATH="/opt/bareun/lib"  # 라이브러리 경로 설정

# Bareun 서비스 백그라운드로 실행
# nohup: 로그아웃 후에도 명령이 계속 실행되도록 함
# &: 명령을 백그라운드로 실행
!BAREUN_ROOT="/opt/bareun" LD_LIBRARY_PATH="/opt/bareun/lib" nohup /opt/bareun/bin/bareun&

In [ ]:
# Bareun 프로세스 실행 확인
# ps -ef: 모든 프로세스 상세 정보 출력
# grep bareun: bareun 문자열이 포함된 라인만 필터링
!ps -ef | grep bareun

In [ ]:
# API 키 등록
# -reg 옵션: API 키 등록
!BAREUN_ROOT="/opt/bareun" LD_LIBRARY_PATH="/opt/bareun/lib" /opt/bareun/bin/bareun -reg koba-YEVHS7Q-VDSUWIY-XCIS3OQ-LWD7WHA

In [ ]:
# Bareun Python 패키지(bareunpy) 설치 또는 업데이트
# -U: 이미 설치된 패키지를 최신 버전으로 업그레이드
!pip install -U bareunpy

In [17]:
# 필요한 라이브러리 임포트
import sys  # 시스템 관련 함수 및 변수 제공
import google.protobuf.text_format as tf  # 프로토콜 버퍼 텍스트 포맷 처리
from bareunpy import Tagger  # 형태소 분석기
from bareunpy import Tokenizer  # 토크나이저
from collections import defaultdict  # 기본값이 있는 딕셔너리

# Bareun API 초기화
API_KEY="koba-YEVHS7Q-VDSUWIY-XCIS3OQ-LWD7WHA"  # API 키 설정
tagger = Tagger(API_KEY, 'localhost', 5656)  # 형태소 분석기 객체 생성 (localhost:5656 서버에 연결)
tokenizer = Tokenizer(API_KEY, 'localhost', 5656)  # 토크나이저 객체 생성

In [ ]:
# pandas 라이브러리 임포트 (데이터 처리)
import pandas as pd

# Excel 파일 불러오기
file_path = '교권_news.xlsx'  # 분석할 Excel 파일 경로

# DataFrame으로 Excel 파일 로드
# sheet_name=0: 첫 번째 시트 선택
df = pd.read_excel(file_path, sheet_name=0)

# 데이터 미리보기 출력
print("Excel file loaded successfully. Preview:")  # 성공 메시지
print(df.head())  # 처음 5개 행 출력

# 데이터프레임 기본 정보 출력
print("\nDataFrame info:")
print(f"Shape: {df.shape}")  # 행과 열의 수 (shape)
print(f"Columns: {df.columns.tolist()}")  # 컬럼명 목록

In [ ]:
# 제목/본문에서 명사 추출 함수 (한 글자 명사 제외)
def extract_nouns_from_texts(texts, tagger):
    all_nouns = []
    for text in texts:
        if not isinstance(text, str):
            continue
        nouns = tagger.tags([text]).nouns()
        nouns = [noun for noun in nouns if len(noun) > 1]  # 한 글자 명사 제외
        all_nouns.extend(nouns)
    return all_nouns

# 제목에서 명사 추출
title_texts = df['제목'].dropna().tolist()
title_nouns = extract_nouns_from_texts(title_texts, tagger)

# 본문에서 명사 추출
content_texts = df['본문'].dropna().tolist()
content_nouns = extract_nouns_from_texts(content_texts, tagger)

# 제목+본문 전체 명사 합치기
all_nouns = title_nouns + content_nouns

# 빈도수 집계
from collections import Counter
noun_counts = Counter(all_nouns)
top_nouns = noun_counts.most_common(20)

print("상위 20개 명사와 빈도:")
for noun, count in top_nouns:
    print(f"{noun}: {count}")

In [ ]:
# 워드클라우드 시각화

from wordcloud import WordCloud  # 워드클라우드 생성을 위한 라이브러리 임포트
import matplotlib.pyplot as plt  # 그래프(이미지) 출력을 위한 라이브러리 임포트

# 워드클라우드 객체 생성
# - font_path: 한글 폰트 경로 지정 (한글 깨짐 방지)
# - width, height: 워드클라우드 이미지 크기 지정
# - background_color: 배경색 지정
wordcloud = WordCloud(font_path='NanumGothic.ttf', width=800, height=400, background_color='white')

# top_nouns(상위 20개 명사와 빈도) 정보를 워드클라우드에 적용
# dict(top_nouns): [('단어', 빈도), ...] 형태를 딕셔너리로 변환
wordcloud.generate_from_frequencies(dict(top_nouns))

# 워드클라우드 이미지를 그릴 도화지(figure) 생성, 크기 지정
plt.figure(figsize=(10, 5))

# 워드클라우드 이미지를 화면에 표시
# interpolation='bilinear'는 이미지를 부드럽게 보이게 함
plt.imshow(wordcloud, interpolation='bilinear')

# x, y축 눈금(테두리) 숨기기
plt.axis('off')

# 실제로 이미지를 화면에 출력
plt.show()

In [ ]:
# 막대그래프 시각화
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 라이브러리 임포트

# top_nouns는 [('단어', 빈도), ...] 형태의 리스트임
# zip(*top_nouns)를 사용하면 단어와 빈도수를 각각 분리해서 labels, values에 저장
labels, values = zip(*top_nouns)

# 그래프의 크기를 가로 10, 세로 5로 설정
plt.figure(figsize=(10, 5))

# 막대그래프(bar chart) 그리기: labels(단어)가 x축, values(빈도수)가 y축
plt.bar(labels, values)

# x축의 단어들이 겹치지 않도록 45도 기울여서 표시
plt.xticks(rotation=45)

# 그래프의 제목 설정
plt.title('상위 20개 명사 빈도')

# 그래프를 화면에 출력
plt.show()

In [ ]:
import matplotlib.pyplot as plt  # 그래프(시각화)를 그리기 위한 라이브러리
import networkx as nx            # 네트워크 그래프를 그리기 위한 라이브러리
from collections import Counter, defaultdict  # 데이터 집계와 기본값 딕셔너리 사용

# 1. 각 문서별로 명사 리스트 만들기
doc_nouns = []  # 각 문서의 명사 리스트를 저장할 빈 리스트 생성
for idx, row in df.iterrows():  # 데이터프레임의 각 행(문서)에 대해 반복
    title = row['제목']         # 제목 컬럼에서 텍스트 추출
    content = row['본문']       # 본문 컬럼에서 텍스트 추출
    nouns = []                  # 한 문서의 명사들을 저장할 리스트
    if isinstance(title, str):  # 제목이 문자열이면
        # 제목에서 명사만 추출(길이 2 이상만), 리스트에 추가
        nouns += [n for n in tagger.tags([title]).nouns() if len(n) > 1]
    if isinstance(content, str):  # 본문이 문자열이면
        # 본문에서 명사만 추출(길이 2 이상만), 리스트에 추가
        nouns += [n for n in tagger.tags([content]).nouns() if len(n) > 1]
    doc_nouns.append(nouns)      # 한 문서의 명사 리스트를 전체 리스트에 추가

# 2. 전체 문서에서 상위 20개 명사만 추출
all_nouns = [n for nouns in doc_nouns for n in nouns]  # 모든 문서의 명사를 하나의 리스트로 합침
top_nouns = set([n for n, _ in Counter(all_nouns).most_common(20)])  # 가장 많이 나온 20개 명사만 set으로 저장

# 3. 명사들의 동시 출현(같은 문서에 함께 등장) 횟수 세기
co_occur = defaultdict(int)  # (명사1, 명사2) 쌍의 동시 출현 횟수를 저장할 딕셔너리
for nouns in doc_nouns:      # 각 문서의 명사 리스트에 대해
    filtered = [n for n in nouns if n in top_nouns]  # 상위 20개 명사만 남김
    for i in range(len(filtered)):                   # 명사 리스트에서
        for j in range(i+1, len(filtered)):          # 모든 명사 쌍에 대해
            pair = tuple(sorted([filtered[i], filtered[j]]))  # 명사 쌍을 알파벳순으로 정렬하여 튜플로 만듦
            co_occur[pair] += 1                      # 해당 쌍이 함께 등장한 횟수 1 증가

# 4. 네트워크 그래프 생성 및 시각화
G = nx.Graph()  # 빈 그래프 객체 생성
for (a, b), cnt in co_occur.items():  # 동시 출현 쌍과 등장 횟수에 대해 반복
    if cnt >= 2:                      # 2번 이상 함께 등장한 쌍만 그래프에 추가
        G.add_edge(a, b, weight=cnt)  # 명사 a, b를 연결하는 간선 추가(가중치는 등장 횟수)

plt.figure(figsize=(10, 6))           # 그래프 크기 설정
pos = nx.spring_layout(G, seed=42)    # 노드의 위치를 spring layout으로 자동 배치(랜덤시드 고정)
nx.draw(
    G, pos, with_labels=True,         # 노드에 라벨(명사) 표시
    node_size=4800,                   # 노드 크기
    node_color='skyblue',             # 노드 색상
    font_size=15,                     # 글씨 크기
    font_family='NanumGothic'         # 한글 폰트 지정(한글 깨짐 방지)
)
plt.title('단어 네트워크(상위 20개 명사)')  # 그래프 제목
plt.show()                              # 그래프 화면에 출력